# 03 — Clean V2 Model Walkthrough

Tujuan: menjalankan **satu historical fold** sendiri supaya `model.fit()` dan PR-AUC tidak terasa seperti angka yang muncul dari Codex.

Ini notebook edukasi. Gunakan hanya historical-development table yang memang sudah pernah dibuka untuk research. Jangan gunakan forward outcome vault.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import average_precision_score, roc_auc_score

HISTORICAL_TABLE = Path(r"CHANGE_ME")
LABEL_COL = "CHANGE_ME"
SESSION_COL = "signal_session_index"

In [ ]:
if not HISTORICAL_TABLE.exists():
    raise FileNotFoundError("Set HISTORICAL_TABLE ke prepared historical-development table.")
df = pd.read_parquet(HISTORICAL_TABLE) if HISTORICAL_TABLE.suffix.lower() == ".parquet" else pd.read_csv(HISTORICAL_TABLE)
print(df.shape)

## Frozen Clean V2 feature order

In [ ]:
V2_FEATURES = [
    "xs_rank_close_return_5",
    "xs_rank_close_return_20",
    "xs_rank_atr14_over_close",
    "xs_rank_close_position_20",
    "xs_rank_distance_high_20_atr",
    "xs_rank_distance_low_20_atr",
    "xs_rank_distance_high_60_atr",
    "xs_rank_distance_low_60_atr",
    "xs_rank_relative_volume_20",
    "xs_rank_log_regular_value_relative_20",
    "market_primary_liquid_count",
    "market_breadth_return_5_positive",
    "market_breadth_return_20_positive",
    "market_median_close_return_5",
    "market_median_close_return_20",
    "market_median_atr14_over_close",
    "market_median_close_position_20",
    "market_median_relative_volume_20",
    "market_median_log_regular_value_relative_20",
    "market_relative_close_return_5",
    "market_relative_close_return_20",
    "market_relative_atr14_over_close",
    "market_relative_close_position_20",
    "market_relative_relative_volume_20",
    "market_relative_log_regular_value_relative_20",
]
missing = [c for c in [*V2_FEATURES, LABEL_COL, SESSION_COL] if c not in df.columns]
if missing:
    raise KeyError(f"Missing required columns: {missing}")
print("feature count:", len(V2_FEATURES))

## Ambil satu frozen fold

V2F1:
- train: session 1–504
- purge: 505–524
- validation: 525–624

Purged sessions **tidak** dimasukkan ke training atau validation.

In [ ]:
train = df[df[SESSION_COL].between(1, 504)].copy()
val = df[df[SESSION_COL].between(525, 624)].copy()

print("train:", len(train), "rows")
print("val:", len(val), "rows")
print("purge gap:", 20, "sessions")

## Bangun estimator Clean V2

In [ ]:
preprocess = ColumnTransformer(
    [(
        "numeric",
        SimpleImputer(strategy="median", add_indicator=True, keep_empty_features=True),
        V2_FEATURES,
    )],
    remainder="drop",
)

model = Pipeline([
    ("preprocess", preprocess),
    ("model", HistGradientBoostingClassifier(
        learning_rate=0.05,
        max_iter=200,
        max_leaf_nodes=31,
        l2_regularization=1.0,
        random_state=42,
    )),
])

model

In [ ]:
X_train = train[V2_FEATURES]
y_train = pd.to_numeric(train[LABEL_COL], errors="raise").astype(int)
X_val = val[V2_FEATURES]
y_val = pd.to_numeric(val[LABEL_COL], errors="raise").astype(int)

print("train positive rate:", y_train.mean())
print("val positive rate:", y_val.mean())

model.fit(X_train, y_train)
pred = model.predict_proba(X_val)[:, 1]

print("PR-AUC :", average_precision_score(y_val, pred))
print("ROC-AUC:", roc_auc_score(y_val, pred))

## Baca hasilnya

- **PR-AUC**: seberapa baik model memprioritaskan positive class ketika class balance tidak harus 50/50.
- **ROC-AUC**: ordering positive vs negative secara global.
- Satu fold bukan verdict. Research runner membandingkan paired folds dan gate yang sudah difreeze.
- Jangan tune parameter dari notebook ini berdasarkan hasil satu fold; tujuan notebook ini adalah pemahaman.